# 3.1 算子开发：membase 与 regbase

本节以 FastGelu（y = x / (1 + exp(-1.702 * x))）为例，分别用 membase 和 regbase 两种模式实现同一个算子。

---

## 学习目标

- 理解 Host 侧的分核和 UB 切分逻辑
- 掌握 Kernel 侧 CopyIn / Compute / CopyOut 的逐步实现
- 理解 regbase 模式下寄存器的选择、UB 的辅助作用和 VF 函数实现

---

In [ ]:
!mkdir -p Sources/03.02

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

## 1. membase

Host 侧通过 TilingFunc 完成分核和 UB 切分，将参数下发给 Kernel。

### 1.1 Host 侧

#### 1.1.1 分核逻辑

分核（Data Sharding）是指将待计算的数据均匀划分到多个 AI Core 上并行处理。通用分核策略如下：

**策略一：均分策略**（适用于数据量能被核数整除的场景）

```
blockLength = totalLength / blockDim
Core[i] 的数据范围: [i × blockLength, (i+1) × blockLength)
```

**策略二：余数补齐策略**（适用于数据量不能被核数整除）

```
blockLength = (totalLength + blockDim - 1) / blockDim   // 向上取整
Core[i] 的数据范围: [i × blockLength, min((i+1) × blockLength, totalLength))
最后一个核的数据可能少于 blockLength
```

**策略三：动态切分**（适用于动态 Shape 场景，在 TilingFunc 中根据实际输入大小计算）

本示例采用**策略一**（均分），数据量 16384 能被 8 整除：

<img src="./images/multicore.png" alt="multicore" width="700px">

**FastGelu 示例中的分核实现**：

Host 侧通过 `<<<blockDim, ...>>>` 指定核数，Kernel 侧通过 `GetBlockIdx()` / `GetBlockNum()` 获取当前核编号和总核数，自行计算偏移：

```cpp
// Kernel Init 中的分核代码
uint32_t blockLength = totalLength / GetBlockNum();      // 2048
uint32_t offset = blockLength * GetBlockIdx();            // Core 0:0, Core 1:2048, ...
xGm.SetGlobalBuffer((__gm__ float *)x + offset, blockLength);  // 绑定当前核的数据范围
yGm.SetGlobalBuffer((__gm__ float *)y + offset, blockLength);
```

先写入头文件和常量：

In [ ]:
%%writefile Sources/03.02/fast_gelu_membase.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <cmath>
#include "acl/acl.h"
#include "kernel_operator.h"

using namespace AscendC;
constexpr uint32_t BUFFER_NUM = 2;

#### 1.1.2 UB 切分（TilingFunc）

TilingFunc 的核心任务是：根据算子需要的 buffer 数量和 UB 容量（256KB，实际可用约 248KB，框架预留了约 8KB），反推每块最多能处理多少元素。

**第一步：识别算子需要多少 buffer**

分析 Kernel 的 Compute 函数中使用了哪些 LocalTensor，每类 tensor 各需要占用一块 UB 空间：

<img src="./images/ub_tiling.png" alt="ub tiling" width="700px">

**第二步：按统一类型等效折算**

不同 buffer 可能存放不同类型的元素（float 4B、half 2B、int32 4B、uint8 1B 等），需要统一换算为**字节数**才能与 UB 容量（256KB）进行比较：

```
折算规则：将各类 buffer 的元素数 × 各自类型的字节宽度

  xLocal (float, 4B/t)  = tileLength × 4  bytes
  yLocal (float, 4B/t)  = tileLength × 4  bytes
  tLocal (float, 4B/t)  = tileLength × 4  bytes
  ─────────────────────────────────────
  单份字节总计            = tileLength × 12 bytes
```

如果 buffer 类型不同（例如增加一个 uint8_t 的 mask）：

```
  xLocal (float, 4B/t)   = tileLength × 4  bytes
  yLocal (float, 4B/t)   = tileLength × 4  bytes
  tLocal (float, 4B/t)   = tileLength × 4  bytes
  mask   (uint8, 1B/t)   = tileLength × 1  bytes
  ─────────────────────────────────────
  单份字节总计             = tileLength × 13 bytes
```

**第三步：考虑双缓冲（Double Buffer）**

输入/输出队列需要 BUFFER_NUM=2 份（搬运与计算并行），临时队列仅需 1 份（单次 Compute 内使用）：

```
  inQueueX   = BUFFER_NUM × xLocal = 2 × tileLength × 4 = tileLength × 8   bytes
  outQueueY  = BUFFER_NUM × yLocal = 2 × tileLength × 4 = tileLength × 8   bytes
  tempBuf  =            1 × tLocal = 1 × tileLength × 4 = tileLength × 4   bytes
  ────────────────────────────────────────────────────────────────────
  总 UB 占用 = (2 + 2 + 1) × tileLength × 4 = tileLength × 20 bytes
```

**通用公式**：设算子有 N 类需要双缓冲的 buffer、M 类不需要双缓冲的 buffer，每种 buffer 的字节宽度为 `size_i`：

```
总 UB 占用 = Σ(N类: BUFFER_NUM × tileLength × size_i) + Σ(M类: tileLength × size_i)
```

**第四步：根据 UB 容量反推 tileLength 上限**

ascend950 的 UB 容量为 256KB = 262144 bytes：

```
总占用 = tileLength × 20 ≤ 262144
tileLength ≤ 262144 / 20 = 13107.2
```

即每块最多可处理 13107 个 float。本示例取 tileLength = 128，留有大量余量。

**第五步：根据 tileLength 反推 tileNum**

```
blockLength = totalLength / blockDim = 16384 / 8 = 2048
tileNum = blockLength / (tileLength × BUFFER_NUM) = 2048 / (128 × 2) = 8
```

**TilingFunc 伪代码**：

```cpp
ge::graphStatus TilingFunc(gert::TilingContext* context) {
    auto xDesc = context->GetInputDesc(0);
    uint32_t totalLength = xDesc->GetShape().GetShapeSize();
    MembaseTiling tiling;
    tiling.totalLength = totalLength;
    tiling.tileNum = 8;  // 由上述五步推导得出
    context->SetTilingData(&tiling, sizeof(MembaseTiling));
    return ge::GRAPH_SUCCESS;
}
```

### 1.2 Kernel 侧

Kernel 侧接收 Host 传入的 tiling 参数，执行 GM ↔ UB 搬运和计算。整个 Kernel 遵循三段式流水结构：

<img src="./images/process_pipeline.png" alt="process pipeline" width="700px">

Kernel 中贯穿三段的核心操作有三种：

| 操作 | API | 方向 | 作用 |
|------|-----|------|------|
| **DataCopy** | `DataCopy(dst, src, len)` | GM ↔ UB | 将 len 个元素从 src 搬运到 dst，双向通用 |
| **EnQue** | `q.EnQue(tensor)` | 队列入队 | 将 LocalTensor 放入队列尾部，供下游消费 |
| **DeQue** | `q.DeQue<T>()` | 队列出队 | 从队列头部取出一个 LocalTensor |

**DataCopy 的细节**：
- 支持 GlobalTensor ↔ LocalTensor 双向搬运
- `xGm[offset]` 语法从 GlobalTensor 中切片，定位第 offset 个元素
- 搬运是异步的——DataCopy 调用后数据可能尚未到达，需通过 EnQue/DeQue 配对确保同步

#### 1.2.1 Init

Init 完成 GlobalTensor 地址绑定和队列分配，具体代码：

In [ ]:
%%writefile -a Sources/03.02/fast_gelu_membase.asc

struct MembaseTiling { uint32_t totalLength; uint32_t tileNum; };

class KernelFastGelu {
public:
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y,
                                  uint32_t totalLength, uint32_t tileNum) {
        this->blockLength = totalLength / GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = blockLength / tileNum / BUFFER_NUM;
        uint32_t offset = blockLength * GetBlockIdx();
        xGm.SetGlobalBuffer((__gm__ float*)x + offset, blockLength);
        yGm.SetGlobalBuffer((__gm__ float*)y + offset, blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, tileLength * sizeof(float));
        pipe.InitBuffer(outQueueY, BUFFER_NUM, tileLength * sizeof(float));
        pipe.InitBuffer(tempBuf, tileLength * sizeof(float));
    }

    __aicore__ inline void Process() {
        int32_t loopCount = tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i); Compute(i); CopyOut(i);
        }
    }

#### 1.2.2 CopyIn：GM → UB

CopyIn 将 Global Memory 中的第 progress 个 tile 搬运到 UB。核心是 **AllocTensor → DataCopy → EnQue** 三步。

**EnQue/DeQue 队列机制详解**：

<img src="./images/queue_states.png" alt="queue states" width="700px">

**关键规则**：
- EnQue 和 DeQue 必须**严格配对**——每入队一次，就必须出队一次
- 队列容量 = BUFFER_NUM，超出会阻塞
- CopyIn 的 EnQue 必须对应 Compute 的 DeQue，确保搬运完成后才能开始计算

In [ ]:
%%writefile -a Sources/03.02/fast_gelu_membase.asc

private:
    __aicore__ inline void CopyIn(int32_t progress) {
        LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
        DataCopy(xLocal, xGm[progress * tileLength], tileLength);
        inQueueX.EnQue(xLocal);
    }

#### 1.2.3 Compute：UB 上计算

Compute 从输入队列取出数据，执行计算，结果放入输出队列。核心是 **DeQue → 计算 → EnQue → FreeTensor**。

<img src="./images/fastgelu_chain.png" alt="fastgelu chain" width="700px">

**FreeTensor 的细节**：
- 释放后该 UB 内存可以被 `AllocTensor` 重新分配
- 不释放会导致队列满、后续 `AllocTensor` 失败
- 每 `AllocTensor` 必须对应一次 `FreeTensor`

In [ ]:
%%writefile -a Sources/03.02/fast_gelu_membase.asc

    __aicore__ inline void Compute(int32_t progress) {
        LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        LocalTensor<float> yLocal = outQueueY.AllocTensor<float>();
        LocalTensor<float> tLocal = tempBuf.Get<float>();

        // FastGelu: y = x / (1 + exp(-1.702 * x))
        Muls(tLocal, xLocal, 1.702f, tileLength);      // t = 1.702 * x
        Neg(tLocal, tLocal, tileLength);               // t = -1.702 * x
        Exp(tLocal, tLocal, tileLength);               // t = exp(-1.702 * x)
        Duplicate(yLocal, 1.0f, tileLength);           // y = 1
        Add(yLocal, yLocal, tLocal, tileLength);       // y = 1+exp(-1.702*x)
        Div(yLocal, xLocal, yLocal, tileLength);       // y = x/(1+exp(-1.702*x))

        outQueueY.EnQue<float>(yLocal);
        inQueueX.FreeTensor(xLocal);
    }

#### 1.2.4 CopyOut：UB → GM

CopyOut 将计算结果写回 Global Memory，关闭 GM → UB → GM 的完整循环。

<img src="./images/copyout_flow.png" alt="copyout flow" width="400px">

In [ ]:
%%writefile -a Sources/03.02/fast_gelu_membase.asc

    __aicore__ inline void CopyOut(int32_t progress) {
        LocalTensor<float> yLocal = outQueueY.DeQue<float>();
        DataCopy(yGm[progress * tileLength], yLocal, tileLength);
        outQueueY.FreeTensor(yLocal);
    }

    TPipe pipe;
    TQue<TPosition::VECIN, BUFFER_NUM> inQueueX;
    TQue<TPosition::VECOUT, BUFFER_NUM> outQueueY;
    TBuf<> tempBuf;
    GlobalTensor<float> xGm, yGm;
    uint32_t blockLength, tileNum, tileLength;
};

__vector__ __global__ __aicore__ void fast_gelu_membase(GM_ADDR x, GM_ADDR y,
                                               MembaseTiling tiling) {
    KernelFastGelu op;
    op.Init(x, y, tiling.totalLength, tiling.tileNum);
    op.Process();
}

#### 1.2.5 Host main：数据准备 + 调用 + 验证

In [ ]:
%%writefile -a Sources/03.02/fast_gelu_membase.asc

int32_t main() {
    constexpr uint32_t totalLength = 8 * 2048;
    aclInit(nullptr); aclrtSetDevice(0);
    aclrtStream stream; aclrtCreateStream(&stream);

    float *xHost, *yHost;
    aclrtMallocHost((void**)&xHost, totalLength * sizeof(float));
    aclrtMallocHost((void**)&yHost, totalLength * sizeof(float));
    for (uint32_t i = 0; i < totalLength; i++) {
        xHost[i] = -4.0f + i * 8.0f / totalLength;
    }

    void *xDev, *yDev;
    aclrtMalloc(&xDev, totalLength * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(&yDev, totalLength * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDev, totalLength * sizeof(float), xHost,
                totalLength * sizeof(float), ACL_MEMCPY_HOST_TO_DEVICE);

    MembaseTiling tiling = {totalLength, 8};
    fast_gelu_membase<<<8, nullptr, stream>>>((uint8_t*)xDev, (uint8_t*)yDev, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(yHost, totalLength * sizeof(float), yDev,
                totalLength * sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST);

    bool pass = true;
    for (uint32_t i = 0; i < totalLength; i++) {
        if (fabsf(yHost[i] - xHost[i] / (1.0f + expf(-1.702f * xHost[i]))) > 1e-4f) {
            pass = false; break;
        }
    }
    printf("membase FastGelu: %s\n", pass ? "PASSED" : "FAILED");

    aclrtFree(xDev); aclrtFree(yDev);
    aclrtFreeHost(xHost); aclrtFreeHost(yHost);
    aclrtDestroyStream(stream); aclrtResetDevice(0); aclFinalize();
    return 0;
}

In [ ]:
!bisheng Sources/03.02/fast_gelu_membase.asc --npu-arch=dav-3510 -o Sources/03.02/fast_gelu_membase -lm && echo "Build Successful"

In [ ]:
!./Sources/03.02/fast_gelu_membase

---

## 2. regbase

regbase 的 Host 侧和 Kernel 侧分核/搬运逻辑与 membase 大致相似，唯一区别在 Compute：计算从 UB 上的 SIMD 指令下沉到 `__simd_vf__` 函数中，直接操作 VF 寄存器。

### 2.1 寄存器选择

| 寄存器 | 说明 |
|------|------|
| `RegTensor<T>` | 单向量寄存器，容量 = `GetVecLen() / sizeof(T)` |
| `RegTensor<T, RegTraitNumTwo>` | 双寄存器版本 |
| `MaskReg` | 掩码寄存器 |

选择原则：一次处理的元素数 ≤ 寄存器容量用 `RegTraitNumOne`；最后一块不足向量长度时用 `UpdateMask` + 部分掩码。

### 2.2 UB 辅助作用

regbase 中 UB 不再是计算载体，而是数据中转站：

<img src="./images/regbase_ub_transit.png" alt="regbase ub transit" width="500px">

### 2.3 VF 函数实现

VF 函数（`__simd_vf__`）是 regbase 的核心，遵循固定的四步模板。下面以 FastGelu 为例，先拆解通用写法，再给出完整代码。

**第一步：确定每次处理的元素数**

硬件一次向量操作能处理的字节数由 `GetVecLen()` 返回。除以元素类型宽度即得每次操作的元素数：

```
vecLen = GetVecLen() / sizeof(float)        // 例如 256 / 4 = 64 个 float
repeat = tileLength / vecLen                // 例如 128 / 64 = 2 次加载
                                             // 若不能整除，最后一块用掩码处理
```

**第二步：按块读取数据（Load）**

VF 函数通过 UB 物理地址指针访问数据，每次 `LoadAlign` 从 UB 中加载 `vecLen` 个元素到一个 RegTensor：

<img src="./images/vf_load_store.png" alt="vf load store" width="550px">

```cpp
for (uint16_t i = 0; i < repeat; ++i) {
    LoadAlign(xReg, xAddr + i * vecLen);    // 第 i 块：加载 vecLen 个元素
```

**第三步：掩码控制（Mask）**

`MaskReg` 控制向量指令中哪些元素参与计算。默认用 `MaskPattern::ALL` 全部生效。当最后一块数据不足 `vecLen` 时，需要用部分掩码：

```cpp
// 数据刚好整除 vecLen → 全开掩码
MaskReg mask = CreateMask<float, MaskPattern::ALL>();

// 最后一块不足 vecLen → 部分掩码（例如只处理前 N 个元素）
MaskReg partialMask = CreateMask<float, MaskPattern::SEQUENCE>();
partialMask = UpdateMask<float>(remainingElements);
```

VF 指令带上 mask 参数后，只有 mask 为 true 的元素位置才会被修改。

**第四步：寄存器内计算与搬出（Compute + Store）**

```cpp
    // 计算：所有 VF 指令操作 RegTensor，带 mask
    Muls(tReg, xReg, 1.702f, mask);        // t = 1.702 * x
    Neg(tReg, tReg, mask);                 // t = -1.702 * x
    Exp(tReg, tReg, mask);                 // t = exp(-1.702 * x)
    Adds(tReg, tReg, 1.0f, mask);          // t = 1 + exp(-1.702 * x)
    Div(tReg, xReg, tReg, mask);           // t = x / (1+exp(-x))

    // 搬出：将寄存器结果写回 UB
    StoreAlign(yAddr + i * vecLen, tReg, mask);
}
```

**第五步：内存屏障**

```cpp
LocalMemBar<MemType::VEC_STORE, MemType::VEC_LOAD>();
```

当后续的 Load 依赖前面 Store 的结果时（如多 tile 迭代），必须插入内存屏障保证顺序。

**完整的 FastGelu VF 函数**：

In [ ]:
%%writefile Sources/03.02/fast_gelu_regbase.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <cmath>
#include "acl/acl.h"
#include "kernel_operator.h"

using namespace AscendC;
using namespace AscendC::Reg;
constexpr uint32_t BUFFER_NUM = 2;

// VF 函数：操作寄存器实现 FastGelu
template <typename T>
__simd_vf__ inline void FastGeluVF(__ubuf__ T* xAddr, __ubuf__ T* yAddr,
                                     uint32_t vecLen, uint32_t repeat) {
    MaskReg mask = CreateMask<T, MaskPattern::ALL>();
    RegTensor<T> xReg, tReg;

    for (uint16_t i = 0; i < repeat; ++i) {
        LoadAlign(xReg, xAddr + i * vecLen);         // 1. Load: UB → Reg
        Muls(tReg, xReg, 1.702f, mask);              // 2. t = 1.702 * x
        Neg(tReg, tReg, mask);                       // 3. Compute
        Exp(tReg, tReg, mask);
        Adds(tReg, tReg, 1.0f, mask);
        Div(tReg, xReg, tReg, mask);
        StoreAlign(yAddr + i * vecLen, tReg, mask);  // 3. Store: Reg → UB
    }
    LocalMemBar<MemType::VEC_STORE, MemType::VEC_LOAD>();  // 4. Barrier
}

struct RegbaseTiling { uint32_t totalLength; uint32_t tileNum; };

class KernelFastGeluRegbase {
public:
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y,
                                  uint32_t totalLength, uint32_t tileNum) {
        this->blockLength = totalLength / GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = blockLength / tileNum / BUFFER_NUM;
        this->vecLen = GetVecLen() / sizeof(float);
        uint32_t offset = blockLength * GetBlockIdx();
        xGm.SetGlobalBuffer((__gm__ float *)x + offset, blockLength);
        yGm.SetGlobalBuffer((__gm__ float *)y + offset, blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, tileLength * sizeof(float));
        pipe.InitBuffer(outQueueY, BUFFER_NUM, tileLength * sizeof(float));
    }

    __aicore__ inline void Process() {
        for (int32_t i = 0; i < tileNum * BUFFER_NUM; i++)
            { CopyIn(i); Compute(i); CopyOut(i); }
    }

private:
    __aicore__ inline void CopyIn(int32_t p) {
        auto xL = inQueueX.AllocTensor<float>();
        DataCopy(xL, xGm[p * tileLength], tileLength);
        inQueueX.EnQue(xL);
    }
    __aicore__ inline void Compute(int32_t p) {
        auto xL = inQueueX.DeQue<float>();
        auto yL = outQueueY.AllocTensor<float>();
        FastGeluVF<float>(
            reinterpret_cast<__ubuf__ float*>(xL.GetPhyAddr()),
            reinterpret_cast<__ubuf__ float*>(yL.GetPhyAddr()),
            vecLen, tileLength / vecLen);
        outQueueY.EnQue<float>(yL);
        inQueueX.FreeTensor(xL);
    }
    __aicore__ inline void CopyOut(int32_t p) {
        auto yL = outQueueY.DeQue<float>();
        DataCopy(yGm[p * tileLength], yL, tileLength);
        outQueueY.FreeTensor(yL);
    }
    TPipe pipe;
    TQue<TPosition::VECIN, BUFFER_NUM> inQueueX;
    TQue<TPosition::VECOUT, BUFFER_NUM> outQueueY;
    GlobalTensor<float> xGm, yGm;
    uint32_t blockLength, tileNum, tileLength, vecLen;
};

__vector__ __global__ __aicore__ void fast_gelu_regbase(GM_ADDR x, GM_ADDR y,
                                               RegbaseTiling tiling) {
    KernelFastGeluRegbase op;
    op.Init(x, y, tiling.totalLength, tiling.tileNum);
    op.Process();
}

VF 函数注意事项：

- `__simd_vf__`：标记函数在向量单元执行
- `__ubuf__ T*`：UB 物理地址，连接 AI Core 侧和 VF 侧
- `GetVecLen()`：一次向量操作字节数，用于计算 repeat
- `LocalMemBar`：多 tile 迭代时必须插入，保证 Store 完成后才 Load
- 掩码末尾处理：最后一块不足向量长度时用 `UpdateMask`

In [ ]:
%%writefile -a Sources/03.02/fast_gelu_regbase.asc

int32_t main() {
    constexpr uint32_t totalLength = 8 * 2048;
    aclInit(nullptr); aclrtSetDevice(0);
    aclrtStream stream; aclrtCreateStream(&stream);
    float *xHost, *yHost;
    aclrtMallocHost((void**)&xHost, totalLength * sizeof(float));
    aclrtMallocHost((void**)&yHost, totalLength * sizeof(float));
    for (uint32_t i = 0; i < totalLength; i++)
        xHost[i] = -4.0f + i * 8.0f / totalLength;
    void *xDev, *yDev;
    aclrtMalloc(&xDev, totalLength * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc(&yDev, totalLength * sizeof(float), ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(xDev, totalLength*sizeof(float), xHost,
                totalLength*sizeof(float), ACL_MEMCPY_HOST_TO_DEVICE);
    RegbaseTiling tiling = {totalLength, 8};
    fast_gelu_regbase<<<8, nullptr, stream>>>((uint8_t*)xDev, (uint8_t*)yDev, tiling);
    aclrtSynchronizeStream(stream);
    aclrtMemcpy(yHost, totalLength*sizeof(float), yDev,
                totalLength*sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST);
    bool pass = true;
    for (uint32_t i = 0; i < totalLength; i++) {
        if (fabsf(yHost[i] - xHost[i] / (1.0f + expf(-1.702f * xHost[i]))) > 1e-4f)
            { pass = false; break; }
    }
    printf("regbase FastGelu: %s\n", pass ? "PASSED" : "FAILED");
    aclrtFree(xDev); aclrtFree(yDev);
    aclrtFreeHost(xHost); aclrtFreeHost(yHost);
    aclrtDestroyStream(stream); aclrtResetDevice(0); aclFinalize();
    return 0;
}

In [ ]:
!bisheng Sources/03.02/fast_gelu_regbase.asc --npu-arch=dav-3510 -o Sources/03.02/fast_gelu_regbase -lm && echo "Build Successful"

In [ ]:
!./Sources/03.02/fast_gelu_regbase

## 3. 对比

| 阶段 | membase | regbase |
|------|---------|----------|
| Host 分核 | `<<<8, stream>>>` | 基本一致 |
| Host UB 切分 | TilingFunc 计算，需为 tempBuf 预留 UB | TilingFunc 计算，无需 tempBuf，可省出 UB 空间 |
| Kernel Init | 绑定 GlobalTensor + 分配队列 | 基本一致 |
| CopyIn | `AllocTensor → DataCopy → EnQue` | 基本一致 |
| Compute | `DeQue → SIMD 指令(UB) → EnQue` | `DeQue → VF 函数(Reg) → EnQue` |
| CopyOut | `DeQue → DataCopy → FreeTensor` | 基本一致 |
| 中间结果 | 需要额外 `tempBuf` 分配 UB 空间 | 使用寄存器保存，不占用 UB |

---

下一节：[3.2 调用接口开发](./03.03_api_interface_development.ipynb)